# Build User Features

Этот notebook строит user-level признаки агрегированием по всем CSV-файлам в `data/Data_1`. Используется stage-based подход: сначала создается компактный parquet-слой транзакций, затем считаются отдельные feature stages, а в конце все объединяется в итоговую таблицу `user_features`.


In [20]:
from contextlib import contextmanager
from pathlib import Path
from time import perf_counter
import re

import numpy as np
import pandas as pd
import polars as pl

ROOT = Path('.')
DATA_DIR = ROOT / 'data' / 'Data_1'
REFERENCE_FILE = ROOT / 'trans600.csv'

ARTIFACTS_DIR = ROOT / 'artifacts'
INTERMEDIATE_DIR = ARTIFACTS_DIR / 'intermediate'
FEATURES_DIR = INTERMEDIATE_DIR / 'user_feature_stages'

PIPELINE_VERSION = 'v2'

TX_LAYER_PATH = INTERMEDIATE_DIR / f'tx_enriched_data1_{PIPELINE_VERSION}.parquet'
BASE_FEATURES_PATH = FEATURES_DIR / f'base_activity_features_{PIPELINE_VERSION}.parquet'
DAILY_FEATURES_PATH = FEATURES_DIR / f'daily_features_{PIPELINE_VERSION}.parquet'
CATEGORY_FEATURES_PATH = FEATURES_DIR / f'category_features_{PIPELINE_VERSION}.parquet'
MCC_FEATURES_PATH = FEATURES_DIR / f'mcc_features_{PIPELINE_VERSION}.parquet'
MERCHANT_FEATURES_PATH = FEATURES_DIR / f'merchant_features_{PIPELINE_VERSION}.parquet'

FINAL_PARQUET_PATH = ARTIFACTS_DIR / 'user_features_data1.parquet'
FINAL_CSV_PATH = ARTIFACTS_DIR / 'user_features_data1.csv'

EXPECTED_COLUMNS = [
    'real_transaction_dttm',
    'party_rk',
    'transaction_rk',
    'merchant_type_code',
    'merchant_nm',
    'category_nm',
]

DT_FORMAT = '%Y-%m-%d %H:%M:%S'
TOP_CATEGORY_LIMIT = 30
TOP_MCC_LIMIT = 30

REBUILD_TX_LAYER = False
REBUILD_FEATURE_STAGES = False
REBUILD_FINAL_DATASET = True

DEDUPLICATE_BY_TRANSACTION_ID = False

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

thread_pool_fn = getattr(pl, 'thread_pool_size', None)
if callable(thread_pool_fn):
    print('Polars thread pool size:', thread_pool_fn())


def log(message: str) -> None:
    print(f'[{pd.Timestamp.now().strftime("%H:%M:%S")}] {message}')


@contextmanager
def timed(label: str):
    started = perf_counter()
    log(f'{label} started')
    try:
        yield
    finally:
        elapsed = perf_counter() - started
        log(f'{label} finished in {elapsed:,.2f} sec')


def collect_stream(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine='streaming')
    except TypeError:
        return lf.collect(streaming=True)


def sink_parquet_stage(label: str, path: Path, lf: pl.LazyFrame, rebuild: bool = False) -> None:
    if path.exists() and not rebuild:
        log(f'Используем существующий stage: {path}')
        return
    with timed(label):
        try:
            lf.sink_parquet(path, compression='zstd', statistics=True)
        except TypeError:
            collect_stream(lf).write_parquet(path, compression='zstd')
        log(f'Stage parquet saved: {path}')


def sanitize_token(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r'\s+', '_', value)
    value = re.sub(r'[^\w]+', '_', value, flags=re.UNICODE)
    value = re.sub(r'_+', '_', value).strip('_')
    return value or 'unknown'


def build_unique_feature_name_map(values: list, prefix: str) -> dict:
    mapping = {}
    seen = {}
    for raw_value in values:
        base_name = f'{prefix}{sanitize_token(raw_value)}'
        suffix = seen.get(base_name, 0)
        feature_name = base_name if suffix == 0 else f'{base_name}_{suffix}'
        seen[base_name] = suffix + 1
        mapping[raw_value] = feature_name
    return mapping


def extend_with_required(values: list, required_values: list) -> list:
    result = []
    seen = set()
    for value in values + required_values:
        if value not in seen:
            result.append(value)
            seen.add(value)
    return result


def weekday_name_expr(column_name: str, alias: str = 'dominant_weekday') -> pl.Expr:
    return (
        pl.when(pl.col(column_name) == 1).then(pl.lit('Monday'))
        .when(pl.col(column_name) == 2).then(pl.lit('Tuesday'))
        .when(pl.col(column_name) == 3).then(pl.lit('Wednesday'))
        .when(pl.col(column_name) == 4).then(pl.lit('Thursday'))
        .when(pl.col(column_name) == 5).then(pl.lit('Friday'))
        .when(pl.col(column_name) == 6).then(pl.lit('Saturday'))
        .when(pl.col(column_name) == 7).then(pl.lit('Sunday'))
        .otherwise(pl.lit(None, dtype=pl.Utf8))
        .alias(alias)
    )


Polars thread pool size: 16


## Промежуточные артефакты

Пайплайн устроен stage-based и сохраняет на диске несколько parquet-слоев:

- `tx_enriched_data1.parquet` - компактный транзакционный слой
- `base_activity_features.parquet` - базовые агрегаты и временные профили
- `daily_features.parquet` - дневные поведенческие признаки
- `category_features.parquet` - категориальные доли и category entropy
- `mcc_features.parquet` - MCC-признаки и mcc entropy
- `merchant_features.parquet` - merchant-level признаки

Каждый stage можно пересчитать отдельно: это ускоряет user-level сборку признаков, не пересчитывая все с нуля при upstream-очистке или проверке отдельных агрегатов.


In [21]:
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Папка с данными не найдена: {DATA_DIR}')

csv_files = sorted(DATA_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'В папке {DATA_DIR} не найдено ни одного CSV-файла')

if not REFERENCE_FILE.exists():
    raise FileNotFoundError(f'Референсный файл не найден: {REFERENCE_FILE}')

reference_columns = pl.read_csv(REFERENCE_FILE, n_rows=0).columns
if reference_columns != EXPECTED_COLUMNS:
    raise ValueError(f'Неожиданные колонки в {REFERENCE_FILE}: {reference_columns}')

log(f'Найдено CSV-файлов: {len(csv_files)}')
log(f'Первые 5 файлов: {[path.name for path in csv_files[:5]]}')
log(f'Референсные колонки транзакций: {reference_columns}')

schema_overrides = {
    'real_transaction_dttm': pl.Utf8,
    'party_rk': pl.Utf8,
    'transaction_rk': pl.Utf8,
    'merchant_type_code': pl.Utf8,
    'merchant_nm': pl.Utf8,
    'category_nm': pl.Utf8,
}

with timed('Build compact transaction parquet'):
    if TX_LAYER_PATH.exists() and not REBUILD_TX_LAYER:
        log(f'Используем существующий parquet: {TX_LAYER_PATH}')
    else:
        raw_lf = pl.scan_csv(
            [str(path) for path in csv_files],
            has_header=True,
            schema_overrides=schema_overrides,
            null_values=['', 'null', 'None'],
        )

        input_columns = raw_lf.collect_schema().names()
        if input_columns != EXPECTED_COLUMNS:
            raise ValueError(f'Неожиданные колонки во входных CSV: {input_columns}')

        prepared_lf = raw_lf.select(EXPECTED_COLUMNS).with_columns(
            pl.col('real_transaction_dttm').str.strptime(pl.Datetime, format=DT_FORMAT, strict=False),
            pl.col('party_rk').cast(pl.Utf8),
            pl.col('transaction_rk').cast(pl.Utf8),
            pl.coalesce([pl.col('merchant_nm').cast(pl.Utf8), pl.lit('unknown_merchant')]).alias('merchant_nm'),
            pl.coalesce([pl.col('category_nm').cast(pl.Utf8), pl.lit('unknown')]).alias('category_nm'),
            pl.col('merchant_type_code').cast(pl.Int32, strict=False).fill_null(-1).alias('merchant_type_code'),
        )

        if DEDUPLICATE_BY_TRANSACTION_ID:
            prepared_lf = prepared_lf.unique(subset=['transaction_rk'], keep='any')

        tx_layer_lf = prepared_lf.select(
            'party_rk',
            'transaction_rk',
            'merchant_nm',
            'category_nm',
            'merchant_type_code',
            'real_transaction_dttm',
        ).with_columns(
            pl.col('real_transaction_dttm').dt.truncate('1d').alias('txn_date'),
            pl.col('real_transaction_dttm').dt.strftime('%Y-%m').alias('month_key'),
            pl.col('real_transaction_dttm').dt.truncate('1w').alias('week_start'),
            pl.col('real_transaction_dttm').dt.hour().cast(pl.Int8).alias('hour'),
            pl.col('real_transaction_dttm').dt.weekday().cast(pl.Int8).alias('weekday_num'),
        )

        try:
            tx_layer_lf.sink_parquet(TX_LAYER_PATH, compression='zstd', statistics=True)
        except TypeError:
            collect_stream(tx_layer_lf).write_parquet(TX_LAYER_PATH, compression='zstd')
        log(f'Parquet сохранен: {TX_LAYER_PATH}')

tx_lf = pl.scan_parquet(TX_LAYER_PATH)


[23:38:12] Найдено CSV-файлов: 50
[23:38:12] Первые 5 файлов: ['transaction_10_new.csv', 'transaction_11_new.csv', 'transaction_12_new.csv', 'transaction_13_new.csv', 'transaction_14_new.csv']
[23:38:12] Референсные колонки транзакций: ['real_transaction_dttm', 'party_rk', 'transaction_rk', 'merchant_type_code', 'merchant_nm', 'category_nm']
[23:38:12] Build compact transaction parquet started
[23:39:12] Parquet сохранен: artifacts\intermediate\tx_enriched_data1_v2.parquet
[23:39:12] Build compact transaction parquet finished in 59.84 sec


## Базовая диагностика

Диагностика запускается на compact transaction layer. Здесь три ключевые метрики: `row_count` - размер всего слоя, `unique_users` - число уникальных пользователей, `unique_transactions` - число уникальных `transaction_rk`.


In [22]:
with timed('Basic diagnostics'):
    diag_df = collect_stream(tx_lf.select(
        pl.len().alias('row_count'),
        pl.col('party_rk').n_unique().alias('unique_users'),
        pl.col('transaction_rk').n_unique().alias('unique_transactions'),
    ))

    null_share_df = collect_stream(tx_lf.select([
        pl.col(col).is_null().mean().alias(f'{col}_null_share')
        for col in tx_lf.collect_schema().names()
    ]))

display(diag_df)
display(null_share_df)


[23:54:59] Basic diagnostics started
[23:55:51] Basic diagnostics finished in 52.17 sec


row_count,unique_users,unique_transactions
u32,u32,u32
49999999,1217027,49999999


party_rk_null_share,transaction_rk_null_share,merchant_nm_null_share,category_nm_null_share,merchant_type_code_null_share,real_transaction_dttm_null_share,txn_date_null_share,month_key_null_share,week_start_null_share,hour_null_share,weekday_num_null_share
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Top категорий и top MCC

Для user-level признаков отдельно выбираются top-30 самых частотных категорий и MCC. Для `unknown` используется отдельный bucket, чтобы это значение не смешивалось с `other`.


In [23]:
with timed('Load top category values'):
    raw_top_categories = collect_stream(
        tx_lf.group_by('category_nm')
        .len()
        .sort('len', descending=True)
        .head(TOP_CATEGORY_LIMIT)
        .select('category_nm')
    )['category_nm'].to_list()

top_category_values = [value for value in raw_top_categories if value != 'unknown']
category_unknown_present = collect_stream(
    tx_lf.filter(pl.col('category_nm') == 'unknown').select(pl.len().alias('cnt'))
)['cnt'][0] > 0
if category_unknown_present:
    top_category_values = extend_with_required(top_category_values, ['unknown'])

with timed('Load top MCC values'):
    raw_top_mcc = collect_stream(
        tx_lf.group_by('merchant_type_code')
        .len()
        .sort('len', descending=True)
        .head(TOP_MCC_LIMIT)
        .select('merchant_type_code')
    )['merchant_type_code'].to_list()

top_mcc_values = [int(value) for value in raw_top_mcc if int(value) != -1]
mcc_unknown_present = collect_stream(
    tx_lf.filter(pl.col('merchant_type_code') == -1).select(pl.len().alias('cnt'))
)['cnt'][0] > 0
mcc_bucket_values = [str(value) for value in top_mcc_values]
if mcc_unknown_present:
    mcc_bucket_values = extend_with_required(mcc_bucket_values, ['unknown'])

category_share_name_map = build_unique_feature_name_map(top_category_values + ['other'], 'cat_share_')
mcc_share_name_map = build_unique_feature_name_map(mcc_bucket_values + ['other'], 'mcc_share_')

log(f'Top category buckets: {top_category_values}')
log(f'Top MCC buckets: {mcc_bucket_values}')


[23:55:52] Load top category values started
[23:55:53] Load top category values finished in 1.13 sec
[23:55:53] Load top MCC values started
[23:55:53] Load top MCC values finished in 0.23 sec
[23:55:53] Top category buckets: ['Супермаркеты', 'Фаст Фуд', 'Транспорт', 'Разные товары', 'Топливо', 'Рестораны', 'Аптеки', 'Дом/Ремонт', 'Развлечения', 'Сервисные услуги', 'Автоуслуги', 'Связь/Телеком', 'Одежда/Обувь', 'Красота', 'Медицинские услуги', 'Госсборы', 'Аренда авто', 'Частные услуги', 'НКО', 'Финансовые услуги', 'Цветы', 'Книги', 'Животные', 'Образование', 'Кино', 'Турагентства', 'Ж/д билеты', 'Авиабилеты', 'Спорттовары', 'unknown']
[23:55:53] Top MCC buckets: ['5411', '5814', '5499', '3990', '4131', '5912', '5541', '5921', '5812', '4111', '5300', '4121', '5200', '7995', '5331', '5993', '7542', '5964', '4900', '5816', '3991', '5815', '5311', '5462', '7512', '4814', '4816', '5399', '5977', '5533', 'unknown']


## Построение feature stages

Каждый stage parquet агрегируется отдельно на уровне `party_rk`. Это нужно и для читаемости, и для переиспользования: проще считать count-признаки рядом с временным профилем, а не держать все в одном pipeline.


In [24]:
hour_count_cols = [f'__hour_{hour:02d}_cnt' for hour in range(24)]
weekday_count_cols = [f'__weekday_{weekday}_cnt' for weekday in range(1, 8)]


def build_base_activity_lf() -> pl.LazyFrame:
    agg_exprs = [
        pl.len().alias('txn_cnt'),
        pl.col('transaction_rk').n_unique().alias('unique_transaction_cnt'),
        pl.col('txn_date').drop_nulls().n_unique().alias('active_days_cnt'),
        pl.col('month_key').drop_nulls().n_unique().alias('active_months_cnt'),
        pl.col('week_start').drop_nulls().n_unique().alias('active_weeks_cnt'),
        pl.col('real_transaction_dttm').min().alias('first_txn_dttm'),
        pl.col('real_transaction_dttm').max().alias('last_txn_dttm'),
    ]
    agg_exprs.extend([
        pl.col('hour').eq(hour).fill_null(False).cast(pl.UInt32).sum().alias(f'__hour_{hour:02d}_cnt')
        for hour in range(24)
    ])
    agg_exprs.extend([
        pl.col('weekday_num').eq(weekday).fill_null(False).cast(pl.UInt32).sum().alias(f'__weekday_{weekday}_cnt')
        for weekday in range(1, 8)
    ])

    return (
        tx_lf.group_by('party_rk')
        .agg(agg_exprs)
        .with_columns(
            (pl.col('last_txn_dttm') - pl.col('first_txn_dttm')).dt.total_days().fill_null(0).cast(pl.Int32).alias('activity_span_days'),
            pl.when(pl.col('active_days_cnt') > 0).then(pl.col('txn_cnt') / pl.col('active_days_cnt')).otherwise(0.0).alias('avg_txn_per_active_day'),
            pl.when(pl.col('active_months_cnt') > 0).then(pl.col('txn_cnt') / pl.col('active_months_cnt')).otherwise(0.0).alias('avg_txn_per_month_active'),
            pl.when(pl.col('active_weeks_cnt') > 0).then(pl.col('txn_cnt') / pl.col('active_weeks_cnt')).otherwise(0.0).alias('avg_txn_per_week_active'),
            pl.concat_list([pl.col(col) for col in hour_count_cols]).list.arg_max().cast(pl.Int8).alias('dominant_hour'),
            (pl.concat_list([pl.col(col) for col in weekday_count_cols]).list.arg_max() + 1).cast(pl.Int8).alias('dominant_weekday_num'),
            pl.sum_horizontal([pl.col(f'__hour_{hour:02d}_cnt') for hour in range(0, 6)]).alias('__night_cnt'),
            pl.sum_horizontal([pl.col(f'__hour_{hour:02d}_cnt') for hour in range(6, 12)]).alias('__morning_cnt'),
            pl.sum_horizontal([pl.col(f'__hour_{hour:02d}_cnt') for hour in range(12, 18)]).alias('__afternoon_cnt'),
            pl.sum_horizontal([pl.col(f'__hour_{hour:02d}_cnt') for hour in range(18, 24)]).alias('__evening_cnt'),
            pl.sum_horizontal([pl.col('__weekday_1_cnt'), pl.col('__weekday_2_cnt'), pl.col('__weekday_3_cnt'), pl.col('__weekday_4_cnt'), pl.col('__weekday_5_cnt')]).alias('__weekday_cnt'),
            pl.sum_horizontal([pl.col('__weekday_6_cnt'), pl.col('__weekday_7_cnt')]).alias('__weekend_cnt'),
        )
        .with_columns(
            pl.when(pl.col('txn_cnt') > 0).then(pl.col('__weekend_cnt') / pl.col('txn_cnt')).otherwise(0.0).alias('share_weekend_txn'),
            pl.when(pl.col('txn_cnt') > 0).then(pl.col('__weekday_cnt') / pl.col('txn_cnt')).otherwise(0.0).alias('share_weekday_txn'),
            pl.when(pl.col('txn_cnt') > 0).then(pl.col('__night_cnt') / pl.col('txn_cnt')).otherwise(0.0).alias('share_night_txn'),
            pl.when(pl.col('txn_cnt') > 0).then(pl.col('__morning_cnt') / pl.col('txn_cnt')).otherwise(0.0).alias('share_morning_txn'),
            pl.when(pl.col('txn_cnt') > 0).then(pl.col('__afternoon_cnt') / pl.col('txn_cnt')).otherwise(0.0).alias('share_afternoon_txn'),
            pl.when(pl.col('txn_cnt') > 0).then(pl.col('__evening_cnt') / pl.col('txn_cnt')).otherwise(0.0).alias('share_evening_txn'),
            pl.concat_list([
                pl.col('__night_cnt'),
                pl.col('__morning_cnt'),
                pl.col('__afternoon_cnt'),
                pl.col('__evening_cnt'),
            ]).list.arg_max().cast(pl.Int8).alias('__dominant_daypart_idx'),
        )
        .with_columns(
            pl.when(pl.col('__dominant_daypart_idx') == 0).then(pl.lit('night'))
            .when(pl.col('__dominant_daypart_idx') == 1).then(pl.lit('morning'))
            .when(pl.col('__dominant_daypart_idx') == 2).then(pl.lit('afternoon'))
            .otherwise(pl.lit('evening'))
            .alias('dominant_daypart')
        )
        .drop(hour_count_cols + weekday_count_cols + ['__night_cnt', '__morning_cnt', '__afternoon_cnt', '__evening_cnt', '__weekday_cnt', '__weekend_cnt', '__dominant_daypart_idx'])
    )


def build_daily_features_lf() -> pl.LazyFrame:
    return (
        tx_lf.select('party_rk', 'txn_date')
        .drop_nulls()
        .group_by(['party_rk', 'txn_date'])
        .len()
        .rename({'len': 'txn_day_cnt'})
        .group_by('party_rk')
        .agg(
            pl.col('txn_day_cnt').max().alias('max_txn_in_one_day'),
            pl.col('txn_day_cnt').median().alias('median_txn_per_active_day'),
        )
    )


def build_category_features_lf() -> pl.LazyFrame:
    category_counts_lf = (
        tx_lf.select('party_rk', 'category_nm')
        .group_by(['party_rk', 'category_nm'])
        .len()
        .rename({'len': 'category_txn_cnt'})
        .with_columns(
            (pl.col('category_txn_cnt') / pl.col('category_txn_cnt').sum().over('party_rk')).alias('share'),
            pl.when(pl.col('category_nm') == 'unknown').then(pl.lit('unknown'))
            .when(pl.col('category_nm').is_in(top_category_values)).then(pl.col('category_nm'))
            .otherwise(pl.lit('other'))
            .alias('category_bucket'),
        )
    )

    agg_exprs = [
        pl.len().alias('unique_category_cnt'),
        pl.col('share').max().alias('top_category_txn_share'),
        (-(pl.col('share') * pl.col('share').log()).sum()).fill_null(0.0).alias('category_entropy'),
    ]
    agg_exprs.extend([
        pl.col('share').filter(pl.col('category_bucket') == bucket_value).sum().fill_null(0.0).alias(category_share_name_map[bucket_value])
        for bucket_value in top_category_values
    ])
    agg_exprs.append(
        pl.col('share').filter(pl.col('category_bucket') == 'other').sum().fill_null(0.0).alias(category_share_name_map['other'])
    )

    return category_counts_lf.group_by('party_rk').agg(agg_exprs)


def build_mcc_features_lf() -> pl.LazyFrame:
    mcc_counts_lf = (
        tx_lf.select('party_rk', 'merchant_type_code')
        .group_by(['party_rk', 'merchant_type_code'])
        .len()
        .rename({'len': 'mcc_txn_cnt'})
        .with_columns(
            (pl.col('mcc_txn_cnt') / pl.col('mcc_txn_cnt').sum().over('party_rk')).alias('share'),
            pl.when(pl.col('merchant_type_code') == -1).then(pl.lit('unknown'))
            .when(pl.col('merchant_type_code').is_in(top_mcc_values)).then(pl.col('merchant_type_code').cast(pl.Utf8))
            .otherwise(pl.lit('other'))
            .alias('mcc_bucket'),
        )
    )

    agg_exprs = [
        pl.len().alias('unique_mcc_cnt'),
        pl.col('share').max().alias('top_mcc_txn_share'),
        (-(pl.col('share') * pl.col('share').log()).sum()).fill_null(0.0).alias('mcc_entropy'),
    ]
    agg_exprs.extend([
        pl.col('share').filter(pl.col('mcc_bucket') == bucket_value).sum().fill_null(0.0).alias(mcc_share_name_map[bucket_value])
        for bucket_value in mcc_bucket_values
    ])
    agg_exprs.append(
        pl.col('share').filter(pl.col('mcc_bucket') == 'other').sum().fill_null(0.0).alias(mcc_share_name_map['other'])
    )

    return mcc_counts_lf.group_by('party_rk').agg(agg_exprs)


def build_merchant_features_lf() -> pl.LazyFrame:
    return (
        tx_lf.select('party_rk', 'merchant_nm')
        .group_by(['party_rk', 'merchant_nm'])
        .len()
        .rename({'len': 'merchant_txn_cnt'})
        .group_by('party_rk')
        .agg(
            pl.len().alias('unique_merchant_cnt'),
            (pl.col('merchant_txn_cnt').max() / pl.col('merchant_txn_cnt').sum()).alias('top_merchant_txn_share'),
        )
    )


sink_parquet_stage('Build base activity features', BASE_FEATURES_PATH, build_base_activity_lf(), rebuild=REBUILD_FEATURE_STAGES)
sink_parquet_stage('Build daily features', DAILY_FEATURES_PATH, build_daily_features_lf(), rebuild=REBUILD_FEATURE_STAGES)
sink_parquet_stage('Build category features', CATEGORY_FEATURES_PATH, build_category_features_lf(), rebuild=REBUILD_FEATURE_STAGES)
sink_parquet_stage('Build MCC features', MCC_FEATURES_PATH, build_mcc_features_lf(), rebuild=REBUILD_FEATURE_STAGES)
sink_parquet_stage('Build merchant features', MERCHANT_FEATURES_PATH, build_merchant_features_lf(), rebuild=REBUILD_FEATURE_STAGES)


[23:55:54] Build base activity features started
[23:57:15] Stage parquet saved: artifacts\intermediate\user_feature_stages\base_activity_features_v2.parquet
[23:57:15] Build base activity features finished in 81.73 sec
[23:57:15] Build daily features started
[23:57:33] Stage parquet saved: artifacts\intermediate\user_feature_stages\daily_features_v2.parquet
[23:57:33] Build daily features finished in 17.41 sec
[23:57:33] Build category features started
[23:57:45] Stage parquet saved: artifacts\intermediate\user_feature_stages\category_features_v2.parquet
[23:57:45] Build category features finished in 11.84 sec
[23:57:45] Build MCC features started
[23:57:58] Stage parquet saved: artifacts\intermediate\user_feature_stages\mcc_features_v2.parquet
[23:57:58] Build MCC features finished in 13.03 sec
[23:57:58] Build merchant features started
[23:58:17] Stage parquet saved: artifacts\intermediate\user_feature_stages\merchant_features_v2.parquet
[23:58:17] Build merchant features finished in

## Сборка финального user-level датасета

Все stages join-ятся по `party_rk`. После этого считаются diversity-ratio признаки и текстовое название доминирующего дня недели. На выходе каждая строка уже соответствует одному пользователю.


In [25]:
with timed('Assemble final user features parquet'):
    if FINAL_PARQUET_PATH.exists() and not REBUILD_FINAL_DATASET:
        log(f'Используем существующий финальный parquet: {FINAL_PARQUET_PATH}')
    else:
        final_lf = (
            pl.scan_parquet(BASE_FEATURES_PATH)
            .join(pl.scan_parquet(DAILY_FEATURES_PATH), on='party_rk', how='left')
            .join(pl.scan_parquet(CATEGORY_FEATURES_PATH), on='party_rk', how='left')
            .join(pl.scan_parquet(MCC_FEATURES_PATH), on='party_rk', how='left')
            .join(pl.scan_parquet(MERCHANT_FEATURES_PATH), on='party_rk', how='left')
            .with_columns(
                weekday_name_expr('dominant_weekday_num', 'dominant_weekday'),
                pl.when(pl.col('txn_cnt') > 0).then(pl.col('unique_merchant_cnt') / pl.col('txn_cnt')).otherwise(0.0).alias('merchant_diversity_ratio'),
                pl.when(pl.col('txn_cnt') > 0).then(pl.col('unique_category_cnt') / pl.col('txn_cnt')).otherwise(0.0).alias('category_diversity_ratio'),
                pl.when(pl.col('txn_cnt') > 0).then(pl.col('unique_mcc_cnt') / pl.col('txn_cnt')).otherwise(0.0).alias('mcc_diversity_ratio'),
            )
        )

        try:
            final_lf.sink_parquet(FINAL_PARQUET_PATH, compression='zstd', statistics=True)
        except TypeError:
            collect_stream(final_lf).write_parquet(FINAL_PARQUET_PATH, compression='zstd')
        log(f'Final parquet saved: {FINAL_PARQUET_PATH}')

user_features_pl = pl.read_parquet(FINAL_PARQUET_PATH)

assert user_features_pl['party_rk'].n_unique() == user_features_pl.height
assert bool((user_features_pl['unique_transaction_cnt'] <= user_features_pl['txn_cnt']).all())

user_features_check_pl = user_features_pl.with_columns(
    weekday_name_expr('dominant_weekday_num', '_expected_dominant_weekday')
)

consistency_df = user_features_check_pl.select(
    (pl.col('party_rk').n_unique() == pl.len()).alias('one_row_per_user'),
    ((pl.col('dominant_weekday_num') >= 1) & (pl.col('dominant_weekday_num') <= 7)).all().alias('dominant_weekday_num_in_range'),
    (pl.col('_expected_dominant_weekday') == pl.col('dominant_weekday')).all().alias('dominant_weekday_consistent'),
    ((pl.col('share_weekend_txn') >= 0.0) & (pl.col('share_weekend_txn') <= 1.0)).all().alias('share_weekend_in_range'),
    ((pl.col('share_weekday_txn') >= 0.0) & (pl.col('share_weekday_txn') <= 1.0)).all().alias('share_weekday_in_range'),
    pl.when(pl.col('txn_cnt') > 0)
    .then((pl.col('share_weekend_txn') + pl.col('share_weekday_txn') - 1.0).abs() <= 1e-6)
    .otherwise(True)
    .all()
    .alias('weekday_share_sum_close_to_one'),
)

consistency_row = consistency_df.row(0, named=True)
for check_name, check_value in consistency_row.items():
    assert bool(check_value), f'Final consistency check failed: {check_name}'

display(consistency_df)
print('Final shape:', user_features_pl.shape)
print('Column count:', len(user_features_pl.columns))
print('First 25 columns:', user_features_pl.columns[:25])
display(user_features_pl.head())


[23:58:18] Assemble final user features parquet started
[23:58:24] Final parquet saved: artifacts\user_features_data1.parquet
[23:58:24] Assemble final user features parquet finished in 6.55 sec


one_row_per_user,dominant_weekday_num_in_range,dominant_weekday_consistent,share_weekend_in_range,share_weekday_in_range,weekday_share_sum_close_to_one
bool,bool,bool,bool,bool,bool
true,true,true,true,true,true


Final shape: (1217027, 98)
Column count: 98
First 25 columns: ['party_rk', 'txn_cnt', 'unique_transaction_cnt', 'active_days_cnt', 'active_months_cnt', 'active_weeks_cnt', 'first_txn_dttm', 'last_txn_dttm', 'activity_span_days', 'avg_txn_per_active_day', 'avg_txn_per_month_active', 'avg_txn_per_week_active', 'dominant_hour', 'dominant_weekday_num', 'share_weekend_txn', 'share_weekday_txn', 'share_night_txn', 'share_morning_txn', 'share_afternoon_txn', 'share_evening_txn', 'dominant_daypart', 'max_txn_in_one_day', 'median_txn_per_active_day', 'unique_category_cnt', 'top_category_txn_share']


party_rk,txn_cnt,unique_transaction_cnt,active_days_cnt,active_months_cnt,active_weeks_cnt,first_txn_dttm,last_txn_dttm,activity_span_days,avg_txn_per_active_day,avg_txn_per_month_active,avg_txn_per_week_active,dominant_hour,dominant_weekday_num,share_weekend_txn,share_weekday_txn,share_night_txn,share_morning_txn,share_afternoon_txn,share_evening_txn,dominant_daypart,max_txn_in_one_day,median_txn_per_active_day,unique_category_cnt,top_category_txn_share,category_entropy,cat_share_супермаркеты,cat_share_фаст_фуд,cat_share_транспорт,cat_share_разные_товары,cat_share_топливо,cat_share_рестораны,cat_share_аптеки,cat_share_дом_ремонт,cat_share_развлечения,cat_share_сервисные_услуги,cat_share_автоуслуги,…,mcc_share_5814,mcc_share_5499,mcc_share_3990,mcc_share_4131,mcc_share_5912,mcc_share_5541,mcc_share_5921,mcc_share_5812,mcc_share_4111,mcc_share_5300,mcc_share_4121,mcc_share_5200,mcc_share_7995,mcc_share_5331,mcc_share_5993,mcc_share_7542,mcc_share_5964,mcc_share_4900,mcc_share_5816,mcc_share_3991,mcc_share_5815,mcc_share_5311,mcc_share_5462,mcc_share_7512,mcc_share_4814,mcc_share_4816,mcc_share_5399,mcc_share_5977,mcc_share_5533,mcc_share_unknown,mcc_share_other,unique_merchant_cnt,top_merchant_txn_share,dominant_weekday,merchant_diversity_ratio,category_diversity_ratio,mcc_diversity_ratio
str,u32,u32,u32,u32,u32,datetime[μs],datetime[μs],i32,f64,f64,f64,i8,i8,f64,f64,f64,f64,f64,f64,str,u32,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,str,f64,f64,f64
"""07e95d24bb488b49697420c27d0351…",2,2,1,1,1,2025-02-03 14:17:04,2025-02-03 14:28:39,0,2.0,2.0,2.0,14,1,0.0,1.0,0.0,0.0,1.0,0.0,"""afternoon""",2,2.0,1,1.0,-0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,0.5,"""Monday""",1.0,0.5,0.5
"""cb450aa7466b337776f0f76debb5a4…",62,62,21,1,5,2025-02-01 16:32:20,2025-02-28 22:11:01,27,2.952381,62.0,12.4,19,4,0.274194,0.725806,0.0,0.16129,0.435484,0.403226,"""afternoon""",9,2.0,14,0.16129,2.386007,0.16129,0.16129,0.145161,0.048387,0.080645,0.112903,0.032258,0.0,0.0,0.0,0.032258,…,0.16129,0.0,0.016129,0.0,0.032258,0.064516,0.0,0.112903,0.129032,0.016129,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016129,0.0,0.0,0.0,0.0,0.016129,0.0,0.032258,0.0,0.032258,0.0,0.225806,36,0.129032,"""Thursday""",0.580645,0.225806,0.33871
"""826ea969a9b6ddc9de5c48bc0cea7e…",4,4,3,1,2,2025-02-11 06:53:44,2025-02-27 07:11:47,16,1.333333,4.0,2.0,6,2,0.0,1.0,0.0,0.75,0.25,0.0,"""morning""",2,1.0,1,1.0,-0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,0.75,"""Tuesday""",0.5,0.25,0.25
"""ac290da8c2f1d18d64c8f0d2c0c8a2…",3,3,3,1,3,2025-02-02 14:03:38,2025-02-28 05:58:21,25,1.0,3.0,1.0,5,2,0.333333,0.666667,0.333333,0.333333,0.333333,0.0,"""night""",1,1.0,2,0.666667,0.636514,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,2,0.666667,"""Tuesday""",0.666667,0.666667,0.666667
"""756690cae8f32d3db05ac30f56d836…",33,33,19,1,5,2025-02-02 13:34:33,2025-02-28 18:00:23,26,1.736842,33.0,6.6,18,2,0.212121,0.787879,0.0,0.363636,0.333333,0.30303,"""morning""",4,2.0,9,0.484848,1.666611,0.484848,0.151515,0.0,0.090909,0.0,0.0,0.090909,0.0,0.030303,0.0,0.0,…,0.151515,0.0,0.0,0.0,0.090909,0.0,0.0,0.0,0.0,0.030303,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.030303,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.242424,24,0.121212,"""Tuesday""",0.727273,0.272727,0.363636


## Экспорт в pandas и сохранение результатов

На этом этапе результат уже user-level, поэтому его удобно дополнительно перевести в `pandas.DataFrame` и сохранить также в CSV.


In [27]:
with timed('Save final files'):
    user_features_pl.write_parquet(FINAL_PARQUET_PATH)
    user_features_pl.write_csv(FINAL_CSV_PATH)
    user_features_df = user_features_pl.to_pandas()

print(type(user_features_pl))
display(user_features_pl.head())
print(user_features_pl.shape)
print('Parquet saved to:', FINAL_PARQUET_PATH)
print('CSV saved to:', FINAL_CSV_PATH)



[00:00:06] Save final files started
[00:00:12] Save final files finished in 5.31 sec
<class 'polars.dataframe.frame.DataFrame'>


party_rk,txn_cnt,unique_transaction_cnt,active_days_cnt,active_months_cnt,active_weeks_cnt,first_txn_dttm,last_txn_dttm,activity_span_days,avg_txn_per_active_day,avg_txn_per_month_active,avg_txn_per_week_active,dominant_hour,dominant_weekday_num,share_weekend_txn,share_weekday_txn,share_night_txn,share_morning_txn,share_afternoon_txn,share_evening_txn,dominant_daypart,max_txn_in_one_day,median_txn_per_active_day,unique_category_cnt,top_category_txn_share,category_entropy,cat_share_супермаркеты,cat_share_фаст_фуд,cat_share_транспорт,cat_share_разные_товары,cat_share_топливо,cat_share_рестораны,cat_share_аптеки,cat_share_дом_ремонт,cat_share_развлечения,cat_share_сервисные_услуги,cat_share_автоуслуги,…,mcc_share_5814,mcc_share_5499,mcc_share_3990,mcc_share_4131,mcc_share_5912,mcc_share_5541,mcc_share_5921,mcc_share_5812,mcc_share_4111,mcc_share_5300,mcc_share_4121,mcc_share_5200,mcc_share_7995,mcc_share_5331,mcc_share_5993,mcc_share_7542,mcc_share_5964,mcc_share_4900,mcc_share_5816,mcc_share_3991,mcc_share_5815,mcc_share_5311,mcc_share_5462,mcc_share_7512,mcc_share_4814,mcc_share_4816,mcc_share_5399,mcc_share_5977,mcc_share_5533,mcc_share_unknown,mcc_share_other,unique_merchant_cnt,top_merchant_txn_share,dominant_weekday,merchant_diversity_ratio,category_diversity_ratio,mcc_diversity_ratio
str,u32,u32,u32,u32,u32,datetime[μs],datetime[μs],i32,f64,f64,f64,i8,i8,f64,f64,f64,f64,f64,f64,str,u32,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,str,f64,f64,f64
"""07e95d24bb488b49697420c27d0351…",2,2,1,1,1,2025-02-03 14:17:04,2025-02-03 14:28:39,0,2.0,2.0,2.0,14,1,0.0,1.0,0.0,0.0,1.0,0.0,"""afternoon""",2,2.0,1,1.0,-0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,0.5,"""Monday""",1.0,0.5,0.5
"""cb450aa7466b337776f0f76debb5a4…",62,62,21,1,5,2025-02-01 16:32:20,2025-02-28 22:11:01,27,2.952381,62.0,12.4,19,4,0.274194,0.725806,0.0,0.16129,0.435484,0.403226,"""afternoon""",9,2.0,14,0.16129,2.386007,0.16129,0.16129,0.145161,0.048387,0.080645,0.112903,0.032258,0.0,0.0,0.0,0.032258,…,0.16129,0.0,0.016129,0.0,0.032258,0.064516,0.0,0.112903,0.129032,0.016129,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016129,0.0,0.0,0.0,0.0,0.016129,0.0,0.032258,0.0,0.032258,0.0,0.225806,36,0.129032,"""Thursday""",0.580645,0.225806,0.33871
"""826ea969a9b6ddc9de5c48bc0cea7e…",4,4,3,1,2,2025-02-11 06:53:44,2025-02-27 07:11:47,16,1.333333,4.0,2.0,6,2,0.0,1.0,0.0,0.75,0.25,0.0,"""morning""",2,1.0,1,1.0,-0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,0.75,"""Tuesday""",0.5,0.25,0.25
"""ac290da8c2f1d18d64c8f0d2c0c8a2…",3,3,3,1,3,2025-02-02 14:03:38,2025-02-28 05:58:21,25,1.0,3.0,1.0,5,2,0.333333,0.666667,0.333333,0.333333,0.333333,0.0,"""night""",1,1.0,2,0.666667,0.636514,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,2,0.666667,"""Tuesday""",0.666667,0.666667,0.666667
"""756690cae8f32d3db05ac30f56d836…",33,33,19,1,5,2025-02-02 13:34:33,2025-02-28 18:00:23,26,1.736842,33.0,6.6,18,2,0.212121,0.787879,0.0,0.363636,0.333333,0.30303,"""morning""",4,2.0,9,0.484848,1.666611,0.484848,0.151515,0.0,0.090909,0.0,0.0,0.090909,0.0,0.030303,0.0,0.0,…,0.151515,0.0,0.0,0.0,0.090909,0.0,0.0,0.0,0.0,0.030303,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.030303,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.242424,24,0.121212,"""Tuesday""",0.727273,0.272727,0.363636


(1217027, 98)
Parquet saved to: artifacts\user_features_data1.parquet
CSV saved to: artifacts\user_features_data1.csv


## Финальная диагностика

Проверяем, что итоговые доли корректно нормализованы, а справочные признаки согласованы с рассчитанными долями категорий и MCC.


In [28]:
display(user_features_df.nlargest(10, 'txn_cnt')[['party_rk', 'txn_cnt', 'unique_transaction_cnt']])
display(user_features_df['txn_cnt'].describe())

cat_share_cols = [col for col in user_features_df.columns if col.startswith('cat_share_')]
mcc_share_cols = [col for col in user_features_df.columns if col.startswith('mcc_share_')]

print('cat_share columns:', cat_share_cols)
print('mcc_share columns:', mcc_share_cols)

cat_share_sum = user_features_df.loc[user_features_df['txn_cnt'] > 0, cat_share_cols].sum(axis=1)
mcc_share_sum = user_features_df.loc[user_features_df['txn_cnt'] > 0, mcc_share_cols].sum(axis=1)

print('Category share sums close to 1:', np.allclose(cat_share_sum, 1.0, atol=1e-6))
print('MCC share sums close to 1:', np.allclose(mcc_share_sum, 1.0, atol=1e-6))
print('unique_transaction_cnt <= txn_cnt:', bool((user_features_df['unique_transaction_cnt'] <= user_features_df['txn_cnt']).all()))

weekday_lookup = {
    1: 'Monday',
    2: 'Tuesday',
    3: 'Wednesday',
    4: 'Thursday',
    5: 'Friday',
    6: 'Saturday',
    7: 'Sunday',
}
weekday_match = user_features_df['dominant_weekday_num'].map(weekday_lookup) == user_features_df['dominant_weekday']
print('dominant_weekday consistency:', bool(weekday_match.all()))

display(cat_share_sum.describe())
display(mcc_share_sum.describe())


,party_rk,txn_cnt,unique_transaction_cnt
30226,2b1f92ba3b38db6236b87947ba6fc6d4f11b920f7e3215...,5041,5041
682298,93079d0751f86b2911517c21f2f7441f98a63a626a9580...,4699,4699
477936,2717e44a76aa2f21307656e43f40bf9ab408d26ecba325...,4548,4548
107907,95afc61ee33440e11099710995827ac55795e189d08a69...,3201,3201
851755,5478f61307405f90048ad91523b258bc8aa6d706526127...,2581,2581
912164,0b8d31375e0f56aaf419b4189088649eaed34c90a32a87...,2511,2511
599525,af3aac75d89ace849fc6c1f68ea2e36b588e9abe31dfe7...,2285,2285
260895,fdf359d92196bb65928a9913eae1ee6afeef5eea198872...,2077,2077
50145,43a5bfa61ec5c8d07b06b1ef5e2edbc41b84a431aebf5e...,2075,2075
149244,b25c59e3d4f6fb32d14830499858c9c115c8322afbe7a8...,1860,1860


count    1.217027e+06
mean     4.108372e+01
std      4.654255e+01
min      1.000000e+00
25%      6.000000e+00
50%      2.600000e+01
75%      6.200000e+01
max      5.041000e+03
Name: txn_cnt, dtype: float64

cat_share columns: ['cat_share_супермаркеты', 'cat_share_фаст_фуд', 'cat_share_транспорт', 'cat_share_разные_товары', 'cat_share_топливо', 'cat_share_рестораны', 'cat_share_аптеки', 'cat_share_дом_ремонт', 'cat_share_развлечения', 'cat_share_сервисные_услуги', 'cat_share_автоуслуги', 'cat_share_связь_телеком', 'cat_share_одежда_обувь', 'cat_share_красота', 'cat_share_медицинские_услуги', 'cat_share_госсборы', 'cat_share_аренда_авто', 'cat_share_частные_услуги', 'cat_share_нко', 'cat_share_финансовые_услуги', 'cat_share_цветы', 'cat_share_книги', 'cat_share_животные', 'cat_share_образование', 'cat_share_кино', 'cat_share_турагентства', 'cat_share_ж_д_билеты', 'cat_share_авиабилеты', 'cat_share_спорттовары', 'cat_share_unknown', 'cat_share_other']
mcc_share columns: ['mcc_share_5411', 'mcc_share_5814', 'mcc_share_5499', 'mcc_share_3990', 'mcc_share_4131', 'mcc_share_5912', 'mcc_share_5541', 'mcc_share_5921', 'mcc_share_5812', 'mcc_share_4111', 'mcc_share_5300', 'mcc_share_4121', 'mcc_sha

count    1.217027e+06
mean     1.000000e+00
std      1.112031e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

count    1.217027e+06
mean     1.000000e+00
std      9.964831e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64